# Longformer + Metadata Fusion -- IMDb Rating Predictor (Standalone)

**Architecture:**
```
Script text -> Longformer CLS (768-dim) -+
                                         |-> Concatenate (800) -> FC(128) -> FC(1) -> Rating
Metadata (Year, Decade, Length) -> FC(32)-+
```

**Before running:**
1. **Runtime -> Change runtime type -> Select GPU** (T4)
2. Upload `scripts.zip` (your ~5204 .txt scripts zipped) to Google Drive root (`MyDrive/`)
3. Upload `movie_lengths.xlsx` to Google Drive root (`MyDrive/`)

Then run all cells top to bottom.

In [ ]:
#@title 1. Install Dependencies & GPU Check
!pip install -q transformers tqdm openpyxl

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print("  DEVICE INFORMATION")
print("=" * 60)
print(f"  Using device : {device}")
if torch.cuda.is_available():
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM         : {gpu_mem:.1f} GB")
else:
    print("  WARNING: No GPU detected!")
    print("  Go to Runtime -> Change runtime type -> Select GPU")
print("=" * 60)

In [ ]:
#@title 2. Mount Drive & Setup Data
import os
import shutil
import zipfile

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = "/content/imdb-predictor"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

DRIVE_ROOT = "/content/drive/MyDrive"

# Copy Excel metadata
excel_src = os.path.join(DRIVE_ROOT, "movie_lengths.xlsx")
if os.path.exists(excel_src):
    shutil.copy2(excel_src, "movie_lengths.xlsx")
    print("[OK] movie_lengths.xlsx copied")
else:
    raise FileNotFoundError("Upload movie_lengths.xlsx to MyDrive/ first!")

# Unzip scripts
scripts_zip = os.path.join(DRIVE_ROOT, "scripts.zip")
if not os.path.exists("scripts"):
    if os.path.exists(scripts_zip):
        print("Extracting scripts.zip ...")
        with zipfile.ZipFile(scripts_zip, 'r') as z:
            z.extractall(".")
        print(f"[OK] Extracted. scripts/ has {len(os.listdir('scripts'))} files")
    else:
        raise FileNotFoundError("Upload scripts.zip to MyDrive/ first!")
else:
    print(f"[OK] scripts/ already exists ({len(os.listdir('scripts'))} files)")

print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
#@title 3. Configuration

# ============================================================
# BASE CONFIG (from config.py)
# ============================================================
EXCEL_FILE = 'movie_lengths.xlsx'
EXCEL_FILES = ['movie_lengths.xlsx']
SCRIPTS_DIR = 'scripts/'

SCRIPT_COL = '.txt Files'
RATING_COL = 'IMDb Rating'
YEAR_COL = 'Year'
DECADE_COL = 'Decade'
MOVIE_NAME_COL = 'Movie name'
MOVIE_LENGTH_COL = 'Movie length'

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42

# ============================================================
# LONGFORMER CONFIG
# ============================================================
MODEL_NAME = "allenai/longformer-base-4096"
MAX_LENGTH = 2048            # safe for free Colab T4/P100
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4         # effective batch = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 5
DROPOUT = 0.3
EARLY_STOPPING_PATIENCE = 3

METADATA_INPUT_DIM = 3
METADATA_EMBED_DIM = 32
FUSION_HIDDEN_DIM = 128

DRIVE_SAVE_PATH = "/content/drive/MyDrive/longformer_imdb_model.pt"

print("[OK] Configuration loaded")
print(f"  Model        : {MODEL_NAME}")
print(f"  Max length   : {MAX_LENGTH}")
print(f"  Batch size   : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"  Epochs       : {EPOCHS}")
print(f"  Save path    : {DRIVE_SAVE_PATH}")

In [ ]:
#@title 4. Preprocessing
import re
import numpy as np


class ScriptPreprocessor:
    """Advanced preprocessing for movie scripts."""

    @staticmethod
    def clean_text(text):
        text = text.lower()
        text = re.sub(r'\[.*?\]', ' ', text)
        text = re.sub(r'\(.*?\)', ' ', text)
        text = re.sub(r'^[A-Z][A-Z\s]+:', ' ', text, flags=re.MULTILINE | re.IGNORECASE)
        text = re.sub(r'\d{1,2}:\d{2}(:\d{2})?', ' ', text)
        text = re.sub(r'\bscene\s*\d+\b', ' ', text, flags=re.IGNORECASE)
        text = re.sub(r'\bint\.?\b|\bext\.?\b', ' ', text, flags=re.IGNORECASE)
        text = re.sub(r"[^a-zA-Z0-9'.,!?\s]", ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    @staticmethod
    def extract_features(raw_text):
        features = {}
        features['char_count'] = len(raw_text)
        words = raw_text.split()
        features['word_count'] = len(words)
        lines = raw_text.split('\n')
        features['line_count'] = len(lines)

        if words:
            features['avg_word_length'] = np.mean([len(w) for w in words])
            features['unique_word_ratio'] = len(set(words)) / len(words)
            long_words = [w for w in words if len(w) >= 8]
            features['long_word_ratio'] = len(long_words) / len(words)
        else:
            features['avg_word_length'] = 0
            features['unique_word_ratio'] = 0
            features['long_word_ratio'] = 0

        sentences = re.split(r'[.!?]+', raw_text)
        sentences = [s.strip() for s in sentences if s.strip()]
        features['sentence_count'] = len(sentences)
        if sentences:
            sent_lengths = [len(s.split()) for s in sentences]
            features['avg_sentence_length'] = np.mean(sent_lengths)
            features['sentence_length_std'] = np.std(sent_lengths)
        else:
            features['avg_sentence_length'] = 0
            features['sentence_length_std'] = 0

        character_lines = re.findall(r'^[A-Z][A-Z\s]+:', raw_text, re.MULTILINE)
        features['dialogue_density'] = len(character_lines) / max(len(lines), 1)
        unique_chars = set([c.strip(':').strip() for c in character_lines])
        features['unique_characters'] = len(unique_chars)

        features['exclamation_ratio'] = raw_text.count('!') / max(features['word_count'], 1) * 100
        features['question_ratio'] = raw_text.count('?') / max(features['word_count'], 1) * 100

        action_brackets = len(re.findall(r'\[.*?\]', raw_text))
        action_parens = len(re.findall(r'\(.*?\)', raw_text))
        features['action_density'] = (action_brackets + action_parens) / max(features['word_count'], 1) * 100

        scene_headings = len(re.findall(r'\b(INT|EXT)\.?\s', raw_text, re.IGNORECASE))
        features['scene_count'] = scene_headings
        features['words_per_scene'] = features['word_count'] / max(scene_headings, 1)

        return features


print("[OK] ScriptPreprocessor defined")

In [ ]:
#@title 5. Data Loader
import os
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder


def load_dataset():
    """Load Excel and script files, extract all features."""
    print("=" * 70)
    print("  LOADING DATASET")
    print("=" * 70)

    excel_files_to_load = EXCEL_FILES if EXCEL_FILES else [EXCEL_FILE]
    print(f"\n>> Loading from {len(excel_files_to_load)} Excel file(s):")

    dataframes = []
    for excel_file in excel_files_to_load:
        if not os.path.exists(excel_file):
            print(f"   [WARNING] File not found: {excel_file}")
            continue
        try:
            df_temp = pd.read_excel(excel_file)
            column_renames = {}
            for col in df_temp.columns:
                col_normalized = ' '.join(col.split())
                if col != col_normalized:
                    column_renames[col] = col_normalized
            if column_renames:
                df_temp = df_temp.rename(columns=column_renames)
            if 'Decaade' in df_temp.columns:
                df_temp = df_temp.rename(columns={'Decaade': 'Decade'})
            df_temp.columns = df_temp.columns.str.strip()

            if RATING_COL in df_temp.columns:
                valid_ratings = pd.to_numeric(df_temp[RATING_COL], errors='coerce').dropna()
                if len(valid_ratings) > 0:
                    print(f"   [OK] {excel_file}: {len(df_temp)} records (Ratings: {valid_ratings.min():.1f}-{valid_ratings.max():.1f})")
            dataframes.append(df_temp)
        except Exception as e:
            print(f"   [ERROR] {excel_file}: {e}")

    if not dataframes:
        raise FileNotFoundError("No valid Excel files found!")

    df = pd.concat(dataframes, ignore_index=True, sort=False)
    df = df.drop_duplicates(subset=[SCRIPT_COL], keep='first')

    expected_cols = [MOVIE_NAME_COL, YEAR_COL, RATING_COL, SCRIPT_COL, DECADE_COL, MOVIE_LENGTH_COL]
    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    print(f"\n>> Combined dataset: {len(df)} unique records")

    scripts_text = []
    script_features = []
    script_files = []
    ratings = []
    years = []
    decades = []
    movie_names = []
    movie_lengths = []

    loaded = 0
    skipped = {'missing': 0, 'short': 0, 'invalid_rating': 0, 'error': 0}

    print(f"\n>> Loading scripts from: {SCRIPTS_DIR}")
    print("   Processing", end="", flush=True)

    for idx, row in df.iterrows():
        if (idx + 1) % 500 == 0:
            print(".", end="", flush=True)

        try:
            rating = float(row[RATING_COL])
            if rating < 0 or rating > 10 or pd.isna(rating):
                skipped['invalid_rating'] += 1
                continue
        except:
            skipped['invalid_rating'] += 1
            continue

        script_file = str(row[SCRIPT_COL]).strip()
        possible_paths = [
            os.path.join(SCRIPTS_DIR, script_file),
            os.path.join(SCRIPTS_DIR, script_file + '.txt'),
            os.path.join(SCRIPTS_DIR, script_file.replace(' ', '-') + '.txt'),
            os.path.join(SCRIPTS_DIR, script_file.replace(' ', '-')),
            os.path.join(SCRIPTS_DIR, script_file.replace(' ', '_') + '.txt'),
            os.path.join(SCRIPTS_DIR, script_file + '.txt.txt'),
        ]
        num_match = re.search(r'(\d+)', script_file)
        if num_match:
            num = num_match.group(1)
            possible_paths.extend([
                os.path.join(SCRIPTS_DIR, f'file-{num}.txt'),
                os.path.join(SCRIPTS_DIR, f'file_{num}.txt'),
                os.path.join(SCRIPTS_DIR, f'file {num}.txt'),
                os.path.join(SCRIPTS_DIR, f'file {num}.txt.txt'),
                os.path.join(SCRIPTS_DIR, f'file{num}.txt'),
            ])

        filepath = None
        for p in possible_paths:
            if os.path.exists(p):
                filepath = p
                break

        if not filepath:
            skipped['missing'] += 1
            continue

        try:
            raw_text = None
            for enc in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    with open(filepath, 'r', encoding=enc) as f:
                        raw_text = f.read()
                    break
                except UnicodeDecodeError:
                    continue

            if not raw_text or len(raw_text) < 1000:
                skipped['short'] += 1
                continue

            cleaned_text = ScriptPreprocessor.clean_text(raw_text)
            features = ScriptPreprocessor.extract_features(raw_text)

            scripts_text.append(cleaned_text)
            script_features.append(features)
            script_files.append(os.path.basename(filepath))
            ratings.append(rating)
            years.append(int(row[YEAR_COL]) if pd.notna(row.get(YEAR_COL)) else 2000)
            decades.append(str(row.get(DECADE_COL, '2000s')))
            movie_names.append(str(row.get(MOVIE_NAME_COL, f'Movie_{idx}')))
            try:
                movie_length = float(row.get(MOVIE_LENGTH_COL)) if pd.notna(row.get(MOVIE_LENGTH_COL)) else None
                if movie_length is not None and movie_length <= 0:
                    movie_length = None
            except (ValueError, TypeError):
                movie_length = None
            movie_lengths.append(movie_length)
            loaded += 1

        except Exception:
            skipped['error'] += 1
            continue

    print(" Done!")
    print(f"\n>> Loading Summary:")
    print(f"   [OK] Successfully loaded: {loaded} scripts")
    print(f"   [X] Skipped: {sum(skipped.values())} total")
    print(f"      - Missing files: {skipped['missing']}")
    print(f"      - Too short (<1KB): {skipped['short']}")
    print(f"      - Invalid rating: {skipped['invalid_rating']}")
    print(f"      - Read errors: {skipped['error']}")

    features_df = pd.DataFrame(script_features)
    features_df['year'] = years
    features_df['movie_length'] = movie_lengths

    le = LabelEncoder()
    features_df['decade_encoded'] = le.fit_transform(decades)

    return scripts_text, np.array(ratings), features_df, movie_names, script_files, le


print("[OK] load_dataset() defined")

In [ ]:
#@title 6. Model & Dataset Classes
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import LongformerModel, PreTrainedTokenizerFast


class LongformerRegressor(nn.Module):
    """Longformer encoder fused with structured metadata for rating prediction."""

    def __init__(
        self,
        model_name=MODEL_NAME,
        dropout=DROPOUT,
        metadata_input_dim=METADATA_INPUT_DIM,
        metadata_embed_dim=METADATA_EMBED_DIM,
        fusion_hidden_dim=FUSION_HIDDEN_DIM,
    ):
        super().__init__()
        self.longformer = LongformerModel.from_pretrained(model_name)
        hidden_size = self.longformer.config.hidden_size  # 768

        self.metadata_fc = nn.Linear(metadata_input_dim, metadata_embed_dim)

        fused_dim = hidden_size + metadata_embed_dim  # 800
        self.regressor = nn.Sequential(
            nn.Linear(fused_dim, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden_dim, 1),
        )

    def forward(self, input_ids, attention_mask, metadata):
        longformer_output = self.longformer(input_ids=input_ids, attention_mask=attention_mask)
        text_embedding = longformer_output.last_hidden_state[:, 0, :]  # CLS
        meta_embedding = F.relu(self.metadata_fc(metadata))
        combined = torch.cat([text_embedding, meta_embedding], dim=1)
        return self.regressor(combined).squeeze(-1)


class MovieScriptDataset(Dataset):
    """Tokenizes scripts on the fly, returns metadata + rating."""

    def __init__(self, texts, ratings, metadata, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.ratings = ratings.astype(np.float32)
        self.metadata = metadata.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "metadata": torch.tensor(self.metadata[idx]),
            "rating": torch.tensor(self.ratings[idx]),
        }


print("[OK] LongformerRegressor and MovieScriptDataset defined")

In [ ]:
#@title 7. Training Functions
import math
import time
import warnings

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import LongformerTokenizerFast, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")


def load_and_split_data():
    """Load dataset and perform 70/15/15 split."""
    scripts_text, ratings, features_df, movie_names, script_files, decade_encoder = load_dataset()

    metadata_cols = ["year", "decade_encoded", "movie_length"]
    metadata_raw = features_df[metadata_cols].copy()
    median_length = metadata_raw["movie_length"].median()
    metadata_raw["movie_length"] = metadata_raw["movie_length"].fillna(median_length)
    metadata_np = metadata_raw.values.astype(np.float32)

    texts_arr = np.array(scripts_text, dtype=object)

    train_texts, temp_texts, train_ratings, temp_ratings, train_meta, temp_meta = train_test_split(
        texts_arr, ratings, metadata_np, test_size=0.30, random_state=RANDOM_STATE,
    )
    val_texts, test_texts, val_ratings, test_ratings, val_meta, test_meta = train_test_split(
        temp_texts, temp_ratings, temp_meta, test_size=0.50, random_state=RANDOM_STATE,
    )

    print(f"\n>> Data Split:")
    print(f"   Train      : {len(train_texts)} samples")
    print(f"   Validation : {len(val_texts)} samples")
    print(f"   Test       : {len(test_texts)} samples")

    return (
        train_texts.tolist(), train_ratings, train_meta,
        val_texts.tolist(), val_ratings, val_meta,
        test_texts.tolist(), test_ratings, test_meta,
    )


def scale_metadata(train_meta, val_meta, test_meta):
    """Fit StandardScaler on training metadata only."""
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_meta)
    val_scaled = scaler.transform(val_meta)
    test_scaled = scaler.transform(test_meta)
    print(f"\n>> Metadata scaling (StandardScaler):")
    print(f"   Train mean: {scaler.mean_}")
    print(f"   Train std : {scaler.scale_}")
    return train_scaled, val_scaled, test_scaled, scaler


def create_dataloaders(
    train_texts, train_ratings, train_meta_s,
    val_texts, val_ratings, val_meta_s,
    test_texts, test_ratings, test_meta_s,
):
    """Create tokenizer, Datasets, and DataLoaders."""
    print(f"\n>> Loading tokenizer: {MODEL_NAME}")
    tokenizer = LongformerTokenizerFast.from_pretrained(MODEL_NAME)

    train_ds = MovieScriptDataset(train_texts, train_ratings, train_meta_s, tokenizer, MAX_LENGTH)
    val_ds = MovieScriptDataset(val_texts, val_ratings, val_meta_s, tokenizer, MAX_LENGTH)
    test_ds = MovieScriptDataset(test_texts, test_ratings, test_meta_s, tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"   Tokenizer loaded  (max_length={MAX_LENGTH})")
    print(f"   Train batches     : {len(train_loader)}")
    print(f"   Validation batches: {len(val_loader)}")
    print(f"   Test batches      : {len(test_loader)}")

    return tokenizer, train_loader, val_loader, test_loader


def train_one_epoch(model, dataloader, optimizer, scheduler, scaler, criterion):
    """One training epoch with mixed precision + gradient accumulation."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    progress = tqdm(dataloader, desc="  Training", leave=False)
    for step, batch in enumerate(progress):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        metadata = batch["metadata"].to(device)
        labels = batch["rating"].to(device)

        with autocast():
            predictions = model(input_ids, attention_mask, metadata)
            loss = criterion(predictions, labels)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        num_batches += 1
        progress.set_postfix(loss=f"{total_loss / num_batches:.4f}")

    torch.cuda.empty_cache()
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate(model, dataloader, criterion):
    """Evaluate model, return loss + regression metrics."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    all_preds = []
    all_labels = []

    progress = tqdm(dataloader, desc="  Evaluating", leave=False)
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        metadata = batch["metadata"].to(device)
        labels = batch["rating"].to(device)

        with autocast():
            predictions = model(input_ids, attention_mask, metadata)
            loss = criterion(predictions, labels)

        total_loss += loss.item()
        num_batches += 1
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(num_batches, 1)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    mse = mean_squared_error(all_labels, all_preds)
    rmse = math.sqrt(mse)
    r2 = r2_score(all_labels, all_preds)

    return avg_loss, mse, rmse, r2


print("[OK] Training functions defined")

In [ ]:
#@title 8. Run Training

# ---- Load data ----
(
    train_texts, train_ratings, train_meta,
    val_texts, val_ratings, val_meta,
    test_texts, test_ratings, test_meta,
) = load_and_split_data()

# ---- Scale metadata ----
train_meta_s, val_meta_s, test_meta_s, metadata_scaler = scale_metadata(
    train_meta, val_meta, test_meta,
)

# ---- Create DataLoaders ----
tokenizer, train_loader, val_loader, test_loader = create_dataloaders(
    train_texts, train_ratings, train_meta_s,
    val_texts, val_ratings, val_meta_s,
    test_texts, test_ratings, test_meta_s,
)

# ---- Build model ----
print(f"\n>> Building LongformerRegressor")
print(f"   Pretrained : {MODEL_NAME}")
print(f"   Dropout    : {DROPOUT}")
print(f"   Meta dims  : {METADATA_INPUT_DIM} -> {METADATA_EMBED_DIM}")
print(f"   Fusion     : {768 + METADATA_EMBED_DIM} -> {FUSION_HIDDEN_DIM} -> 1")

model = LongformerRegressor()
model.to(device)

# ---- Optimizer, scheduler, criterion, scaler ----
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

total_training_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = total_training_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

criterion = nn.MSELoss()
grad_scaler = GradScaler()

print(f"\n>> Training configuration:")
print(f"   Epochs              : {EPOCHS}")
print(f"   Batch size          : {BATCH_SIZE}")
print(f"   Grad accum steps    : {GRAD_ACCUM_STEPS}")
print(f"   Effective batch     : {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   Learning rate       : {LEARNING_RATE}")
print(f"   Weight decay        : {WEIGHT_DECAY}")
print(f"   Total steps         : {total_training_steps}")
print(f"   Warmup steps        : {warmup_steps}")
print(f"   Early stopping      : {EARLY_STOPPING_PATIENCE} epochs patience")
print(f"   Mixed precision     : enabled")

# ---- Training loop ----
best_val_loss = float("inf")
patience_counter = 0

print("\n" + "=" * 60)
print("  TRAINING STARTED")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, grad_scaler, criterion,
    )
    val_loss, val_mse, val_rmse, val_r2 = evaluate(model, val_loader, criterion)

    epoch_time = time.time() - epoch_start

    print(f"\n  Epoch {epoch}/{EPOCHS}  ({epoch_time:.0f}s)")
    print(f"    Train Loss : {train_loss:.4f}")
    print(f"    Val Loss   : {val_loss:.4f}")
    print(f"    Val RMSE   : {val_rmse:.4f}")
    print(f"    Val R\u00b2     : {val_r2:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "best_val_loss": best_val_loss,
            "best_val_rmse": val_rmse,
            "best_val_r2": val_r2,
            "epoch": epoch,
            "metadata_scaler_mean": metadata_scaler.mean_.tolist(),
            "metadata_scaler_scale": metadata_scaler.scale_.tolist(),
            "config": {
                "model_name": MODEL_NAME,
                "max_length": MAX_LENGTH,
                "dropout": DROPOUT,
                "metadata_input_dim": METADATA_INPUT_DIM,
                "metadata_embed_dim": METADATA_EMBED_DIM,
                "fusion_hidden_dim": FUSION_HIDDEN_DIM,
            },
        }

        try:
            torch.save(checkpoint, DRIVE_SAVE_PATH)
            print(f"    >> Best model saved to {DRIVE_SAVE_PATH}")
        except OSError:
            torch.save(checkpoint, "longformer_imdb_model.pt")
            print(f"    >> Best model saved locally")

        try:
            tokenizer.save_pretrained(DRIVE_SAVE_PATH.replace(".pt", "_tokenizer"))
        except OSError:
            tokenizer.save_pretrained("longformer_imdb_tokenizer")

    else:
        patience_counter += 1
        print(f"    >> No improvement ({patience_counter}/{EARLY_STOPPING_PATIENCE})")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n  Early stopping triggered at epoch {epoch}.")
            break

    torch.cuda.empty_cache()

In [ ]:
#@title 9. Final Test Evaluation

print("\n" + "=" * 60)
print("  FINAL EVALUATION ON TEST SET")
print("=" * 60)

# Load best checkpoint
try:
    best_ckpt = torch.load(DRIVE_SAVE_PATH, map_location=device)
except OSError:
    best_ckpt = torch.load("longformer_imdb_model.pt", map_location=device)

model.load_state_dict(best_ckpt["model_state_dict"])
model.to(device)

test_loss, test_mse, test_rmse, test_r2 = evaluate(model, test_loader, criterion)

print(f"\n  Test MSE  : {test_mse:.4f}")
print(f"  Test RMSE : {test_rmse:.4f}")
print(f"  Test R\u00b2   : {test_r2:.4f}")
print(f"\n  Best val loss was {best_ckpt['best_val_loss']:.4f} at epoch {best_ckpt['epoch']}")
print("=" * 60)
print("  TRAINING COMPLETE")
print("=" * 60)